<h1> Gerador do Golden Truth </h1>
Objetivo: Gerar um conjunto de dados com perguntas, respostas; ou seja, o golden truth para ser usada na avaliação do assistente autônomo. </br>
Entrada: base de dados com os textos extraídos dos documentos da RAG, base de dados com informaçòes cadastrais, financeiras de clientes, previsões com score, base de interações. </br>
Saida: dataframe json pronto para ser usado. </br>

<h3> Importação de pacotes e bibliotecas </h3>

In [18]:
import json
from pathlib import Path
import google.generativeai as genai
import pandas as pd
from tqdm import tqdm
import os

<h3> Rotina </h3>

In [19]:
# Insira sua chave de API aqui
GOOGLE_API_KEY = "AQ.Ab8RN6KG8l1nYtGTG_jBfUP-6dRixPbreJZcbcNWBlHwzx5b5Q"
genai.configure(api_key=GOOGLE_API_KEY)

In [20]:
# ==============================================================================
# CONFIGURAÇÃO DE CAMINHOS E MODELO
# ==============================================================================
RAIZ_PROJETO = Path(os.getcwd()).parent

DIR_BASES = RAIZ_PROJETO / "bases_tratadas"
DIR_RESULTADOS = RAIZ_PROJETO / "resultados"

CAMINHO_DOCS_RAG = DIR_BASES / "df_documentos_para_vetorizar.csv"
CAMINHO_CLIENTES = DIR_BASES / "clientes_sem_nulos.csv"
CAMINHO_INTERACOES = DIR_BASES / "interacoes_com_info_data.csv"
CAMINHO_XGBOOST = DIR_RESULTADOS / "xgboost_inferencia_todos.csv"

CAMINHO_SAIDA_GOLDEN = DIR_BASES / "golden_set_completo.csv"

# Verificação opcional para garantir que encontrou a raiz correta
print(f"📁 Raiz do projeto definida em: {RAIZ_PROJETO}")

MODEL_NAME = "gemini-3.1-flash-lite"

# ==============================================================================
# PROMPT PARA GERAR PERGUNTAS DO RAG (DOCUMENTOS TEXTUAIS)
# ==============================================================================
PROMPT_RAG = """
Você é um especialista em validação de sistemas RAG para produtos de crédito e regularização PJ.

Sua tarefa é ler o texto do documento fornecido abaixo e criar 2 pares de (Pergunta, Resposta Esperada, Ground Truth) estritamente baseados nas informações contidas nele.

DOCUMENTO (Arquivo: {nome_arquivo}):
\"\"\"
{texto_documento}
\"\"\"

Siga rigorosamente as instruções:
1. As perguntas devem simular dúvidas reais de analistas de crédito ou especialistas de políticas.
2. A 'resposta_esperada' deve ser completa e natural.
3. O 'ground_truth' deve conter o fato exato/trecho direto extraído do documento.
4. Retorne EXCLUSIVAMENTE em formato JSON válido:

[
  {{
    "pergunta": "Texto da pergunta 1",
    "resposta_esperada": "Resposta completa do assistente",
    "ground_truth": "Fato extraído do documento"
  }},
  {{
    "pergunta": "Texto da pergunta 2",
    "resposta_esperada": "Resposta completa do assistente",
    "ground_truth": "Fato extraído do documento"
  }}
]
"""


# ==============================================================================
# 1. PROCESSADOR DA BASE 1: DOCUMENTOS RAG (Via Gemini API)
# ==============================================================================
def processar_base_rag(model, limite_docs=None) -> list:
  print("\n📄 [1/4] Processando Base RAG (Documentos Textuais)...")
  registros = []

  if not CAMINHO_DOCS_RAG.exists():
    print(f"⚠️ Arquivo não encontrado: {CAMINHO_DOCS_RAG}")
    return registros

  df = pd.read_csv(CAMINHO_DOCS_RAG)
  if limite_docs:
    df = df.head(limite_docs)

  for _, row in tqdm(df.iterrows(), total=len(df)):
    doc_id = str(row.get("id", ""))
    nome_arquivo = str(row.get("nome_arquivo_texto", "desconhecido.txt"))
    texto = str(row.get("texto", ""))

    if len(texto.strip()) < 50:
      continue

    prompt = PROMPT_RAG.format(
        nome_arquivo=nome_arquivo, texto_documento=texto[:3000]
    )

    try:
      response = model.generate_content(
          prompt,
          generation_config=genai.GenerationConfig(
              response_mime_type="application/json", temperature=0.2
          ),
      )
      pares = json.loads(response.text)

      for par in pares:
        registros.append({
            "origem": "RAG_DOCUMENTOS",
            "tema": "Políticas e Manuais RAG",
            "subtema": nome_arquivo,
            "doc_id": doc_id,
            "contexto": texto,
            "pergunta": par.get("pergunta"),
            "resposta_esperada": par.get("resposta_esperada"),
            "ground_truth": par.get("ground_truth"),
        })
    except Exception as e:
      print(f"⚠️ Erro ao processar RAG {nome_arquivo}: {e}")

  return registros


# ==============================================================================
# 2. PROCESSADOR DA BASE 2: CLIENTES (Dados Cadastrais e Financeiros)
# ==============================================================================
def processar_base_clientes(amostra=10) -> list:
  print("\n👤 [2/4] Processando Base de Clientes Cadastrais/Financeiros...")
  registros = []

  if not CAMINHO_CLIENTES.exists():
    print(f"⚠️ Arquivo não encontrado: {CAMINHO_CLIENTES}")
    return registros

  df = pd.read_csv(CAMINHO_CLIENTES).sample(
      n=min(amostra, len(pd.read_csv(CAMINHO_CLIENTES))), random_state=42
  )

  for _, row in df.iterrows():
    # Tenta identificar colunas padrão comuns no dataframe
    cnpj = str(
        row.get("cnpj", row.get("cnpj_cpf", row.get("id_cliente", "N/A")))
    )
    nome = str(
        row.get("nome", row.get("razao_social", f"Cliente CNPJ {cnpj}"))
    )
    faturamento = row.get("faturamento", row.get("receita_anual", "N/A"))
    porte = row.get("porte", row.get("porte_empresa", "N/A"))

    contexto = (
        f"CNPJ: {cnpj} | Razão Social: {nome} | Porte: {porte} | Faturamento:"
        f" {faturamento}"
    )

    # Pergunta 1: Dados Cadastrais
    registros.append({
        "origem": "CADASTRO_CLIENTES",
        "tema": "Consulta Cliente",
        "subtema": "Dados Cadastrais",
        "doc_id": f"CLI_{cnpj}",
        "contexto": contexto,
        "pergunta": f"Quais são as informações cadastrais do cliente CNPJ {cnpj}?",
        "resposta_esperada": (
            f"O cliente {nome} (CNPJ {cnpj}) é classificado como porte {porte}"
            f" e possui faturamento informado de {faturamento}."
        ),
        "ground_truth": (
            f"Razão Social: {nome}, Porte: {porte}, Faturamento: {faturamento}"
        ),
    })

  return registros


# ==============================================================================
# 3. PROCESSADOR DA BASE 3: INTERAÇÕES CANAIS DIGITAIS
# ==============================================================================
def processar_base_interacoes(amostra=10) -> list:
  print("\n💬 [3/4] Processando Base de Interações dos Canais...")
  registros = []

  if not CAMINHO_INTERACOES.exists():
    print(f"⚠️ Arquivo não encontrado: {CAMINHO_INTERACOES}")
    return registros

  df = pd.read_csv(CAMINHO_INTERACOES).sample(
      n=min(amostra, len(pd.read_csv(CAMINHO_INTERACOES))), random_state=42
  )

  for _, row in df.iterrows():
    cnpj = str(
        row.get("cnpj", row.get("id_cliente", row.get("cnpj_cpf", "N/A")))
    )
    canal = str(row.get("canal", row.get("nome_canal", "Canal Digital")))
    data_interacao = str(
        row.get("data", row.get("data_interacao", "Data recente"))
    )
    motivo = str(
        row.get(
            "motivo",
            row.get("descricao", row.get("tipo_interacao", "Atendimento")),
        )
    )

    contexto = (
        f"CNPJ: {cnpj} | Canal: {canal} | Data: {data_interacao} | Motivo/Status:"
        f" {motivo}"
    )

    registros.append({
        "origem": "INTERACOES_CANAIS",
        "tema": "Histórico de Interações",
        "subtema": canal,
        "doc_id": f"INT_{cnpj}",
        "contexto": contexto,
        "pergunta": (
            f"Qual foi o último registro de interação do cliente CNPJ {cnpj} nos"
            " canais?"
        ),
        "resposta_esperada": (
            f"A última interação registrada do cliente CNPJ {cnpj} ocorreu em"
            f" {data_interacao} através do canal {canal}, referente a: {motivo}."
        ),
        "ground_truth": (
            f"Canal: {canal}, Data: {data_interacao}, Motivo: {motivo}"
        ),
    })

  return registros


# ==============================================================================
# 4. PROCESSADOR DA BASE 4: XGBOOST INFERÊNCIA (Scores e Predições)
# ==============================================================================
def processar_base_xgboost(amostra=10) -> list:
  print("\n📊 [4/4] Processando Resultados XGBoost (Scores e Predições)...")
  registros = []

  if not CAMINHO_XGBOOST.exists():
    print(f"⚠️ Arquivo não encontrado: {CAMINHO_XGBOOST}")
    return registros

  df = pd.read_csv(CAMINHO_XGBOOST).sample(
      n=min(amostra, len(pd.read_csv(CAMINHO_XGBOOST))), random_state=42
  )

  for _, row in df.iterrows():
    cnpj = str(
        row.get("cnpj", row.get("id_cliente", row.get("cnpj_cpf", "N/A")))
    )
    score = row.get("score", row.get("score_predito", row.get("proba", "N/A")))
    classe_predita = row.get(
        "predicao",
        row.get("classe", row.get("status_regularizacao", "N/A")),
    )

    contexto = (
        f"CNPJ: {cnpj} | Score Preditivo XGBoost: {score} | Classe/Status:"
        f" {classe_predita}"
    )

    registros.append({
        "origem": "XGBOOST_INFERENCIA",
        "tema": "Score e Predição Preditiva",
        "subtema": "Modelo XGBoost",
        "doc_id": f"XGB_{cnpj}",
        "contexto": contexto,
        "pergunta": (
            "Qual o score de regularização predito pelo modelo XGBoost para o"
            f" cliente CNPJ {cnpj}?"
        ),
        "resposta_esperada": (
            f"De acordo com a inferência do modelo XGBoost, o cliente CNPJ"
            f" {cnpj} possui um score predito de {score}, sendo classificado"
            f" como: {classe_predita}."
        ),
        "ground_truth": f"Score: {score}, Predição: {classe_predita}",
    })

  return registros


# ==============================================================================
# PIPELINE PRINCIPAL DE CONSOLIDAÇÃO
# ==============================================================================
def gerar_golden_set_completo():
  print("🚀 Iniciando Construção do Golden Set Multibases...")

  model = genai.GenerativeModel(MODEL_NAME)

  todos_registros = []

  # 1. Base RAG Documentos (Geração via Gemini)
  todos_registros.extend(processar_base_rag(model))

  # 2. Base Clientes
  todos_registros.extend(processar_base_clientes(amostra=15))

  # 3. Base Interações
  todos_registros.extend(processar_base_interacoes(amostra=15))

  # 4. Base XGBoost
  todos_registros.extend(processar_base_xgboost(amostra=15))

  # Gerar DataFrame Final
  df_golden = pd.DataFrame(todos_registros)

  # Garante ordenação limpa de colunas
  colunas_finais = [
      "origem",
      "tema",
      "subtema",
      "doc_id",
      "contexto",
      "pergunta",
      "resposta_esperada",
      "ground_truth",
  ]
  df_golden = df_golden[colunas_finais]

  # Salva em CSV
  df_golden.to_csv(CAMINHO_SAIDA_GOLDEN, index=False, encoding="utf-8-sig")

  print("\n" + "=" * 60)
  print("✅ GOLDEN SET COMPLETO GERADO COM SUCESSO!")
  print("=" * 60)
  print(f"📊 Total de Exemplos: {len(df_golden)}")
  print("\n📌 Distribuição por Origem de Dados:")
  print(df_golden["origem"].value_counts().to_string())
  print(f"\n💾 Arquivo salvo em: {CAMINHO_SAIDA_GOLDEN}")


📁 Raiz do projeto definida em: c:\Users\roger\Documents\Dataminutes\Site\Portfolio\itau\ds3_mlai_test


<h3> Chamada Principal Main </h3>

In [21]:
%%time
if __name__ == "__main__":
  gerar_golden_set_completo()

🚀 Iniciando Construção do Golden Set Multibases...

📄 [1/4] Processando Base RAG (Documentos Textuais)...


100%|██████████| 5/5 [00:12<00:00,  2.56s/it]



👤 [2/4] Processando Base de Clientes Cadastrais/Financeiros...

💬 [3/4] Processando Base de Interações dos Canais...

📊 [4/4] Processando Resultados XGBoost (Scores e Predições)...

✅ GOLDEN SET COMPLETO GERADO COM SUCESSO!
📊 Total de Exemplos: 55

📌 Distribuição por Origem de Dados:
origem
CADASTRO_CLIENTES     15
INTERACOES_CANAIS     15
XGBOOST_INFERENCIA    15
RAG_DOCUMENTOS        10

💾 Arquivo salvo em: c:\Users\roger\Documents\Dataminutes\Site\Portfolio\itau\ds3_mlai_test\bases_tratadas\golden_set_completo.csv
CPU times: total: 234 ms
Wall time: 13.1 s
